In [1]:
# import gevent.monkey #* Handle errors between requests and grequests
# gevent.monkey.patch_all()

import pandas as pd
import numpy as np
from pprint import pprint
import time
import os
import matplotlib.pyplot as plt
from typing import *
import math

In [43]:
#* Constants
startingDate = "2020-01-01"
endingDate = "2023-12-31"

#* Format des données horraires
startingDateInt = "2020010100"
endingDateInt = "2023123123"
regions_departements = {
    "Auvergne-Rhône-Alpes": [
        1, 3, 7, 15, 26, 38, 42, 43, 63, 69, 73, 74
    ],
    "Bourgogne-Franche-Comté": [
        21, 25, 39, 58, 70, 71, 89, 90
    ],
    "Bretagne": [
        22, 29, 35, 56
    ],
    "Centre-Val de Loire": [
        18, 28, 36, 37, 41, 45
    ],
    # "Corse": [
    #     20
    # ],
    "Grand Est": [
        8, 10, 51, 52, 54, 55, 57, 67, 68, 88
    ],
    "Hauts-de-France": [
        2, 59, 60, 62, 80
    ],
    "Ile-de-France": [
        75, 77, 78, 91, 92, 93, 94, 95
    ],
    "Nouvelle-Aquitaine": [
        16, 17, 19, 23, 24, 33, 40, 47, 64, 79, 86, 87
    ],
    "Normandie": [
        14, 27, 50, 61, 76
    ],
    "Occitanie": [
        9, 11, 12, 30, 31, 32, 34, 46, 48, 65, 66, 81, 82
    ],
    "Pays de la Loire": [
        44, 49, 53, 72, 85
    ],
    "PACA": [
        4, 5, 6, 13, 83, 84
    ]
}


In [68]:
# Constants


APIurl = "https://tabular-api.data.gouv.fr/api/resources/"


#* Colonnes pour les fichier météo en jour
relevantColsDay = ["NUM_POSTE",#* Numéro de poste,
                    "LAT", #* L'attitude
                    "LON", #* Longitude
                    "AAAAMMJJ", #* Date
                    "RR", #* Quantité de précipipation
                    "TN", #* Température minimal à l'abri
                    "TX", #* Température maximal à l'abri
                    "TM", #* Température moyenne à l'abri
                    "TNSOL", #* Température minimal à 10cm du sol
                    "TN50", #* Température minimal à 50cm du dol
                    "DG", #* Durée du gel
                    "FFM", #* Force moyenne des vents à 10m
                    "FF2M", #* Force moyenne des vents à 2m
                    "DRR", #* Durée des précipitations
                ]

#* Colonnes pour les fichiers météo en heure
relevantColsHOR = [ "NUM_POSTE",#* Numéro de poste,
                    "AAAAMMJJHH", #* Date et heure
                    "LAT", #* L'attitude
                    "LON", #* Longitude
                    "RR1", #* Quantitée des précipitations sur l'heure
                    "DRR1", #* Durée des précipitations sur une heure
                    "FF", #* Force moyenne du vent à 10m
                    "FF2", #* Force moyenne du vent à 2m
                    "T", #* Température à l'abri
                    "TD", #* Température au point de rosé
                    "TN", #* Température minimal à l'abri
                    "TX", #* Température maximal à l'abri
                    "DG", #* Durée du gel en mn
                    "TNSOL", #* Température au sol
                    "TN50", #* Température à 50 cm du sol
                    "U", #* Humidité (%)
                    "UN",#* Humidité min (%)
                    "UX", #* Humidité max (%)
                    "DHUMI40", #* durée humidité <=40%
                    "DHUMI80", #* durée humidité <=80%
                    ]

# reponse = requests.get(APIurl+dataIds[0])

# data = pd.DataFrame(reponse.json()['data'])
# data


In [ ]:
#* GetIds from files 
def getIds(fileName):
    ids = []
    with  open(fileName) as fileIds:
        for link in fileIds:
            ids.append(link[-37:-1])
    return ids
meteoDayIds =  getIds("ressourcesLink.txt")
meteoHORIds =  getIds("HORLinks.txt")
nbDep = len(meteoDayIds)

In [ ]:
""" SWAGGER AND PROFILE"""
# swagger = requests.get("https://tabular-api.data.gouv.fr/api/resources/c79aaafe-8017-4d2b-8884-57b5391da5bc/swagger/")
# for l in swagger.iter_lines():
#     print(l)
def getProfile():
    import requests
    reponse = requests.get("https://tabular-api.data.gouv.fr/api/resources/9aad7600-6ba8-47c6-b31b-79c8631d7ae9/profile/")
    print(reponse.json()['profile'])
# getProfile()

In [ ]:
#* Test call API
query = (
"page_size=50"
"&columns=NUM_POSTE,AAAAMMJJ,TN,TX,TM,TNSOL,TN50,DG,FF2M,DRR"
"&AAAAMMJJ__greater="+startingDate+""
"&AAAAMMJJ__less="+endingDate+""
)
reponse = requests.get("https://tabular-api.data.gouv.fr/api/resources/c79aaafe-8017-4d2b-8884-57b5391da5bc/data/?"+query)
pprint(reponse.json())
if reponse.status_code==200:
    data = pd.DataFrame(reponse.json()['data'])
else:
    pprint(reponse.json())
    
data.head()

In [45]:
def getFileNameMet(step:Literal["raw","cleaned"],delta:Literal["ByDay","ByHour"]=None,departementNumber:int=None):
    metoBase = "meteo_"
    number = str(departementNumber).zfill(2)

    return os.path.join(step+"Data","meteo",delta,metoBase+number+".csv")

def getFileNameEnr(step:Literal["raw","cleaned"],region:str=None,year:int=None):
    if step=="raw":
        # print(os.path.join(step+"Data",dataSource,f"eCO2mix_RTE_{region}_Annuel-Definitif_{year}.zip"))
        return os.path.join(step+"Data","energy",f"eCO2mix_RTE_{region.replace(" ","-")}_Annuel-Definitif_{year}.xls")
    else:
        return os.path.join(step+"Data","energy",f"Consommation_{region.replace(" ","-")}.csv")


In [ ]:
#* Get all data :
# baseAPI = "https://tabular-api.data.gouv.fr/api/resources/"
# query = (
#         "/data/?"
#         "page_size=200"
#         "&columns="+",".join(relevantColsDay)+""
#         "&AAAAMMJJ__greater="+startingDate+"" 
#         "&AAAAMMJJ__less="+endingDate+""
#     )

# #* ~3min per departement
# def getDepMeteoData(id:str):
#     import requests

#     isNext = True
    
#     data = pd.DataFrame()
#     totalRequest = baseAPI+id+query

#     #* Loop throw all pages
#     while isNext:
#         reponse = requests.get(totalRequest)

#         if reponse.status_code==200:
#             reponse = reponse.json()

#             if data.shape[0]>0:
#                 data = pd.concat([data,pd.DataFrame(reponse['data'])]) #* Concat all data together
#             else:
#                 data = pd.DataFrame(reponse['data'])

#             if reponse["links"]["next"]:
#                 totalRequest = reponse["links"]["next"]
#                 time.sleep(0.01) #* The API limit rate is 100/seconds
#             else: #* End of the data
#                 isNext = False

#             print(f"\rPage : {reponse['meta']['page']} / {reponse['meta']['total']/reponse['meta']['page_size']} ",end='', flush=True)

#         else:
#             print('ERROR DURING REQUEST',reponse.json())
#             break
#     print('')
#     return data


# def getMeteoData():
#     for dep,meteoId in list(enumerate(meteoIds))[0:1]:

#         print(f"Département {dep} / {len(meteoIds)}")
#         meteoData = getDepMeteoData(meteoId)
#         print("data size ",meteoData.shape)
#         # time.sleep(5)
#         # meteoData.to_csv(getFileName("raw","meteo",dep))

# getMeteoData()

In [ ]:
#* -------------- ASYNCRHONOUS BY DAY--------------------
#* Get all data :
apiPageSizeLimit =200
baseAPI = "https://tabular-api.data.gouv.fr/api/resources/"
query = (
        "/data/?"
        "page_size="+str(apiPageSizeLimit)+""
        "&columns="+",".join(relevantColsDay)+""
        "&AAAAMMJJ__greater="+startingDate+"" 
        "&AAAAMMJJ__less="+endingDate+""
    )
data = pd.DataFrame()

def handleApiResult(reponse):
    global data
    # print("API HANDLER")
    if reponse and reponse.status_code==200:
        reponse = reponse.json()
        # print(f"\rPage : {reponse['meta']['page']} / {reponse['meta']['total']/reponse['meta']['page_size']} ",end='', flush=True)
        # print(len(reponse["data"]))
        data = pd.concat([data,pd.DataFrame(reponse['data'])]) #* Concat all data together

    else:
        print('ERROR DURING REQUEST',reponse.json())

#* ~3min per departement
def getDepMeteoDataAsync(dep,id:str):
    global data
    import grequests
    
    totalRequest = baseAPI+id+query
    #* first request : 
    req = grequests.get(totalRequest)
    reponse = grequests.map([req])[0]
    if reponse.status_code==200:
        reponse = reponse.json()

        data = pd.DataFrame(reponse['data'])
        nbPages = math.floor(reponse["meta"]["total"]/reponse["meta"]["page_size"]) #* We already get the first page
        print(f"First call done, Calls to make :{nbPages} ")
        #* Loop throw all pages
        pendingCall = []
        page = 2
        while page < nbPages+1:
            currentRequest = totalRequest + "&page="+str(page)
            pendingCall.append(grequests.get(currentRequest)) 
            if page%100==0: #* We make call by batch of 100
                results = grequests.map(pendingCall)
                print("")
                # print(f"Results call for dep {dep}: {results}")
                for result in results:
                    handleApiResult(result)
                time.sleep(1) #* The API limit rate is 100/seconds
                pendingCall = []
                # break
                # print('call Made ',page," / ",nbPages)
            print('\rcall Made ',page," / ",nbPages,end='',flush=True)

            page +=1

        results = grequests.map(pendingCall)
        for result in results:
            handleApiResult(result)
        # print(f"Results call for dep {dep}: {data}")  

    else:
        print('ERROR DURING REQUEST',reponse)

    print('')
    return data


def getMeteoData():
    # for dep,meteoId in list(enumerate(meteoIds)):
    for dep in regions_departements['Bretagne']:
        meteoId = meteoDayIds[dep]
        print(f"Département {dep} / {len(meteoDayIds)}")
        meteoData = getDepMeteoDataAsync(dep+1,meteoId)
        print(meteoData.head())
        print("data size ",meteoData.shape)
        # time.sleep(5)
        meteoData.to_csv(getFileName("raw","meteo",delta="ByDay",departementNumber=dep))
getMeteoData()

In [ ]:
#* -------------- ASYNCRHONOUS BY HOUR--------------------
#* Get all data :
apiPageSizeLimit =200
baseAPI = "https://tabular-api.data.gouv.fr/api/resources/"
query = (
        "/data/?"
        "page_size="+str(apiPageSizeLimit)+""
        "&columns="+",".join(relevantColsHOR)+""
        "&AAAAMMJJHH__greater="+startingDateInt+"" 
        "&AAAAMMJJHH__less="+endingDateInt+""
    )
data = pd.DataFrame()

def handleApiResult(reponse):
    global data
    # print("API HANDLER")
    if reponse and reponse.status_code==200:
        reponse = reponse.json()
        # print(f"\rPage : {reponse['meta']['page']} / {reponse['meta']['total']/reponse['meta']['page_size']} ",end='', flush=True)
        data = pd.concat([data,pd.DataFrame(reponse['data'])]) #* Concat all data together

    else:
        print('ERROR DURING REQUEST',reponse.json())

#* ~3min per departement
def getDepMeteoDataAsync(dep,id:str):
    global data
    import grequests
    
    totalRequest = baseAPI+id+query
    print(totalRequest)
    #* first request : 
    req = grequests.get(totalRequest)
    reponse = grequests.map([req])[0]
    if reponse.status_code==200:
        reponse = reponse.json()

        data = pd.DataFrame(reponse['data'])
        nbPages = math.floor(reponse["meta"]["total"]/reponse["meta"]["page_size"]) #* We already get the first page
        print(f"First call done, Calls to make :{nbPages} ")
        #* Loop throw all pages
        pendingCall = []
        page = 2
        while page < nbPages+1:
            currentRequest = totalRequest + "&page="+str(page)
            pendingCall.append(grequests.get(currentRequest)) 
            if page%100==0: #* We make call by batch of 100
                results = grequests.map(pendingCall)
                # print("")
                # print(f"Results call for dep {dep}: {results}")
                for result in results:
                    handleApiResult(result)
                time.sleep(1) #* The API limit rate is 100/seconds
                pendingCall = []
                # break
            # print('\rnb url to call ',len(pendingCall),end='',flush=True)
            print('\rcall Made ',page," / ",nbPages,end='',flush=True)

            page +=1

        results = grequests.map(pendingCall)
        for result in results:
            handleApiResult(result)
        # print(f"Results call for dep {dep}: {results}")  

    else:
        print('ERROR DURING REQUEST',reponse)

    print('')
    return data


def getMeteoData():
    # for dep,meteoId in list(enumerate(meteoIds)):
    for dep in regions_departements['Bretagne']:
        meteoId = meteoHORIds[dep]
        print(f"Département {dep} / {len(meteoDayIds)}")
        meteoData = getDepMeteoDataAsync(dep+1,meteoId)
        print(meteoData.head())
        print("data size ",meteoData.shape)
        # time.sleep(5)
        meteoData.to_csv(getFileName("raw","meteo",delta="ByHour",departementNumber=dep))
getMeteoData()

**Transform Data**

In [ ]:
#* METEO DAY | Use to filter the dowload to only keep relevant informations 

minDate = 20130101
stats = []
for dep in range(1,96):
    depStr = str(dep).zfill(2)
    dataMeteoRaw = pd.read_csv(os.path.join("rawData","meteo","ByDay",f"Q_{depStr}_previous-1950-2024_RR-T-Vent.csv"),sep=";")
    # print(dataMeteoRaw)
    dataDay = dataMeteoRaw[dataMeteoRaw["AAAAMMJJ"]>=minDate][relevantColsDay]
    stats.append(dataDay.isna().sum()/dataDay.shape[0])
    dataDay.to_csv(getFileNameMet("raw","ByDay",dep))

In [ ]:
#* METEO HOR | Use to filter the dowload to only keep relevant informations 

minDate = 2013010100

for dep in range(1,5):
    depStr = str(dep).zfill(2)
    dataMeteoRaw = pd.read_csv(os.path.join("rawData","meteo","ByHour",f"H_{depStr}_previous-2020-2024.csv"),sep=";")
    # print(dataMeteoRaw)
    dataHOR = dataMeteoRaw[dataMeteoRaw["AAAAMMJJHH"]>=minDate][relevantColsHOR]
    print("\r"+depStr,end='')
    dataHOR.to_csv(getFileNameMet("raw","ByHour",dep))

04

In [67]:
#* Correct the datetime

from concurrent.futures import ThreadPoolExecutor

def process_dep(dep):

    depStr = str(dep).zfill(2)
    # dataHOR = pd.read_csv(getFileNameMet("raw","ByHour",dep), index_col=0)
    
    # dataHOR["AAAAMMJJHH"] = pd.to_datetime(dataHOR["AAAAMMJJHH"],dayfirst=True,format="%Y%m%d%H")
    # dataHOR["DATE"] = dataHOR["AAAAMMJJHH"].dt.date
    # dataHOR["HOUR"] = dataHOR["AAAAMMJJHH"].dt.hour
    
    # dataHOR.drop(columns = ["AAAAMMJJHH"], inplace=True)

    # dataHOR.to_csv(getFileNameMet("cleaned","ByHour",departementNumber=dep),index=False)

    dataDay = pd.read_csv(getFileNameMet("cleaned","ByDay",dep), index_col=0)

    dataDay["AAAAMMJJ"] = pd.to_datetime(dataDay["AAAAMMJJ"],dayfirst=True,format="%Y%m%d")
    dataDay["DATE"] = dataDay["AAAAMMJJ"].dt.date
    
    dataDay.drop(columns=["AAAAMMJJ"], inplace=True)
   

    dataDay.to_csv(getFileNameMet("cleaned","ByDay",departementNumber=dep),index=False)

    print("\r"+depStr,end='')

with ThreadPoolExecutor(max_workers=5) as executor:
    executor.map(process_dep, range(3, 5))
    # executor.map(process_dep, range(1, 96))

03

In [ ]:
#* Energy excel to csv and filter

for region in regions_departements.keys():
    yearsData = []
    for year in range(2020,2024):
        try : 
            yearConsommationDf = pd.read_csv(
                    getFileNameEnr("raw", region, year),
                    sep="\t",
                    encoding="latin-1",  # À cause des accents (é, è...)
                    usecols=["Date", "Heures", "Consommation"]
                ).dropna()

            yearsData.append(yearConsommationDf)
        except:
            print(region,year)
    consommationDf = pd.concat(yearsData)
    consommationDf.to_csv(getFileNameEnr("cleaned",region,year),index=False)
    


In [74]:
#* Divide consommation by 1000
for region in regions_departements.keys():
    consommationDf = pd.read_csv(getFileNameEnr("cleaned",region=region))
    # print(consommationDf)
    consommationDf["Consommation"] = consommationDf["Consommation"] / 10
    consommationDf = consommationDf[["Date","Heures","Consommation"]]
    consommationDf.to_csv(getFileNameEnr("cleaned",region=region),index=False)

In [13]:
#* Meteo by Day 

# for dep in range(1,nbDep+1):
for dep in [2]:
    rawData = pd.read_csv(getFileName("raw","meteo",delta="ByDay",departementNumber=dep-1),index_col=0)

    cleanData = rawData.drop(['NUM_POSTE'],axis=1)
    cleanData.groupby(['AAAAMMJJ']).mean() #* Compute the mean by day on differents stations

    cleanData.to_csv(getFileName("cleaned","meteo",delta="ByDay",departementNumber=dep))

# plt.plot(cleanData["AAAAMMJJ"],cleanData["TN"])


**RECUPERATION DES ESTIMATIONS RTE**

In [50]:
rteFileName = "eCO2mix_RTE_Annuel-Definitif_"
yearsDf = []
#* Convert Excel to csv
for year in range(2020,2024):
    rteData = pd.read_csv(os.path.join("rawData","complement",f"{rteFileName}{year}.xls"),sep="\t",encoding="latin-1",usecols=["Date","Heures","Consommation","Prévision J-1","Prévision J"])
    rteData.dropna(inplace=True)
    yearsDf.append(rteData)
    print(f"year : {year}, len : {rteData.shape}")
rteData = pd.concat(yearsDf)
rteData.to_csv(os.path.join("cleanedData","complement",f"rteEstimation.csv"),index=False)

year : 2020, len : (17568, 5)
year : 2021, len : (17520, 5)
year : 2022, len : (17520, 5)
year : 2023, len : (17520, 5)


In [ ]:
yearsDf =[]
for year in range(2020, 2024):
    rteData = pd.read_csv(
        os.path.join("rawData", "complement", f"{rteFileName}{year}.xls"),
        sep="\t",
        encoding="latin-1",
    )

    rteData.columns = rteData.columns.str.strip()

    # Sélectionner les colonnes par position car les colonnes sont mal nommée/corrompu
    rteData = rteData.iloc[:, [1,2, 3, 4, 5]]  # Date, Heures, Conso, Prev J-1, Prev J
    rteData.columns = ["Date", "Heures", "Consommation", "Prévision J-1", "Prévision J"]
    # print(rteData.head(2))

    # Supprimer uniquement les lignes sans Consommation
    rteData.dropna(subset=["Consommation"], inplace=True)
    yearsDf.append(rteData)
    # print(f"year : {year}, len : {rteData.shape}")
rteData = pd.concat(yearsDf)
rteData.to_csv(os.path.join("cleanedData","complement",f"rteEstimation.csv"),index=False)


              Date Heures  Consommation  Prévision J-1  Prévision J
France  2020-01-01  00:00       67068.0        68000.0      65800.0
France  2020-01-01  00:15           NaN        67300.0      65300.0
year : 2020, len : (17568, 5)


C:\Users\felib\AppData\Local\Temp\ipykernel_9260\2114190820.py:3: DtypeWarning: Columns (36,37,38,39) have mixed types. Specify dtype option on import or set low_memory=False.
  rteData = pd.read_csv(


              Date Heures  Consommation  Prévision J-1  Prévision J
France  2021-01-01  00:00       67156.0        67000.0      66800.0
France  2021-01-01  00:15           NaN        66500.0      66300.0
year : 2021, len : (17520, 5)
              Date Heures  Consommation  Prévision J-1  Prévision J
France  2022-01-01  00:00       54615.0        51500.0      54200.0
France  2022-01-01  00:15           NaN        51100.0      53750.0
year : 2022, len : (17520, 5)
              Date Heures  Consommation  Prévision J-1  Prévision J
France  2023-01-01  00:00       47571.0        46600.0      48000.0
France  2023-01-01  00:15           NaN        46050.0      47500.0
year : 2023, len : (17520, 5)


In [66]:
# deps = [str(i).zfill(2) for i in range(1, 96)]  # 01 à 95

fill_rates = []

for dep in range(1, 96):

    path = getFileNameMet("cleaned", "ByDay", dep)
    df = pd.read_csv(path)
    fill_rate = df.notna().mean() * 100  
    fill_rates.append(fill_rate)


# Moyenne sur tous les départements
summary = pd.DataFrame(fill_rates).mean().round(2).sort_values()
summary.name = "Remplissage moyen (%)"

summary

FF2M           1.19
DRR            5.10
TN50           5.87
TNSOL         12.62
FFM           24.49
DG            46.26
TM            48.69
TX            65.34
TN            65.35
RR            98.52
NUM_POSTE    100.00
DATE         100.00
AAAAMMJJ     100.00
Name: Remplissage moyen (%), dtype: float64